In [1]:
# ==============================================================================
# CELL 1: SETUP, IMPORTS & CONFIGURATION
# ==============================================================================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
# Environment Setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# 12 Target Classes matching train.csv
LABEL_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

print(f"Target Device: {DEVICE}")
print(f"Target Labels ({len(LABEL_COLS)}): {LABEL_COLS}")

Target Device: cuda
Target Labels (12): ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


In [2]:
# ==============================================================================
# CELL 2: CUSTOM DATASET WITH SYNTHETIC NOISE INJECTION
# ==============================================================================
class DualPathMedicalDataset(Dataset):
    """
    Handles train.csv containing 58 labeled rows and 4,349 unlabeled rows.
    Injects synthetic noise into binary flags during training to teach Path 2
    how to verify and reject corrupted inputs.
    """
    def __init__(self, df, tokenizer, max_len=256, is_train=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['Report'])

        # 1. Tokenize multilingual radiology report
        encoded = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        # 2. Parse target values & check for NaNs
        raw_targets = row[LABEL_COLS].values.astype(np.float32)
        has_binary = not np.isnan(raw_targets).all()

        # Clean NaNs to zeros for PyTorch tensor conversion
        targets_clean = np.nan_to_num(raw_targets, nan=0.0)
        targets_tensor = torch.tensor(targets_clean, dtype=torch.float32)
        has_binary_mask = torch.tensor([has_binary], dtype=torch.bool)

        # 3. Construct candidate binary vector for Path 2 (Verification Gate)
        candidate_binary = targets_tensor.clone()
        if self.is_train and has_binary:
            # Synthetic noise: Randomly corrupt binary flags 30% of time
            if np.random.rand() < 0.30:
                num_flips = np.random.randint(1, 3)
                flip_indices = np.random.choice(len(LABEL_COLS), size=num_flips, replace=False)
                candidate_binary[flip_indices] = 1.0 - candidate_binary[flip_indices]

        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'targets': targets_tensor,
            'has_binary_mask': has_binary_mask,
            'candidate_binary': candidate_binary
        }

In [3]:
# ==============================================================================
# CELL 3: DUAL-PATH MEDICAL VERIFIER ARCHITECTURE
# ==============================================================================
class DualPathMedicalVerifier(nn.Module):
    def __init__(self, model_name="./LaBSE", num_classes=12):
        super().__init__()
        self.num_classes = num_classes

        # Multilingual Transformer Encoder
        self.text_encoder = AutoModel.from_pretrained(model_name)
        hidden_dim = self.text_encoder.config.hidden_size # 768 for LaBSE

        # Path 1: Text-only Prediction Head (P_report)
        self.report_head = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes) # Outputs 12 raw logits
        )

        # Path 2: Verification Gate [Text Embeddings (768) + Input Binaries (12)] -> Gate [0, 1]
        self.verification_gate = nn.Sequential(
            nn.Linear(hidden_dim + num_classes, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask, candidate_binary=None, has_binary_mask=None):
        # Mean-pooled text embeddings
        outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_embeds = outputs.last_hidden_state.mean(dim=1)

        # Path 1 probabilities
        report_logits = self.report_head(text_embeds)
        p_report = torch.sigmoid(report_logits)

        # Mode A: If no binary input is provided, return pure report predictions
        if candidate_binary is None or has_binary_mask is None:
            return report_logits, p_report

        # Path 2: Compute verification gate
        gate_inputs = torch.cat([text_embeds, candidate_binary], dim=1)
        gate = self.verification_gate(gate_inputs)

        # Reconciled prediction: Trust-weighted combination
        p_reconciled = gate * candidate_binary + (1.0 - gate) * p_report

        # Mode B: Dynamic Routing based on binary presence
        final_probs = torch.where(has_binary_mask, p_reconciled, p_report)

        return report_logits, final_probs

In [4]:
# ==============================================================================
# CELL 4: MASKED BCE LOSS FUNCTION
# ==============================================================================
class MaskedBCEWithLogitsLoss(nn.Module):
    """Computes BCE loss exclusively on non-NaN labeled rows."""
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets, mask):
        loss = self.bce(logits, targets)
        mask_expanded = mask.expand_as(loss)
        masked_loss = (loss * mask_expanded.float()).sum() / (mask_expanded.float().sum() + 1e-8)
        return masked_loss

In [5]:
# ==============================================================================
# CELL 5: 2-PHASE TRAINING LOOP WITH BEST MODEL CHECKPOINTING
# ==============================================================================
import copy
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# File paths for saving local artifacts
MODEL_SAVE_PATH = "best_dual_path_model.pt"
TOKENIZER_SAVE_DIR = "./labse_tokenizer"

# 1. Separate Labeled vs. Unlabeled Data
df = pd.read_csv("/run/media/mahyar/Samsung-990-PRO/Projects/Kaggle/rsna-knee-abnormality-detection/train.csv")
labeled_df = df[df['ACL'].notnull()].reset_index(drop=True)
unlabeled_df = df[df['ACL'].isnull()].reset_index(drop=True)

# Train/Val split on labeled dataset
train_labeled, val_labeled = train_test_split(labeled_df, test_size=0.2, random_state=SEED)

MODEL_NAME = "./LaBSE"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Save tokenizer locally for offline Kaggle use

# Data Loaders
train_ds = DualPathMedicalDataset(train_labeled, tokenizer, is_train=True)
val_ds = DualPathMedicalDataset(val_labeled, tokenizer, is_train=False)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

model = DualPathMedicalVerifier(model_name=MODEL_NAME).to(DEVICE)
criterion = nn.BCELoss()

# Validation Evaluation Function
def evaluate(model, loader, phase=1):
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            targets = batch['targets'].to(DEVICE)
            candidate_binary = batch['candidate_binary'].to(DEVICE)
            has_binary_mask = batch['has_binary_mask'].to(DEVICE)

            if phase == 1:
                _, probs = model(input_ids, attention_mask)
            else:
                _, probs = model(input_ids, attention_mask, candidate_binary, has_binary_mask)

            loss = criterion(probs, targets)
            val_loss += loss.item()

    return val_loss / len(loader)

# ------------------------------------------------------------------------------
# PHASE 1: FINE-TUNE REPORT EXTRACTION HEAD (Path 1)
# ------------------------------------------------------------------------------
print("\n=== PHASE 1: Training Report Extraction Head ===")
for param in model.text_encoder.parameters():
    param.requires_grad = False
optimizer_p1 = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler_p1 = CosineAnnealingLR(optimizer_p1, T_max=30)

best_val_loss_p1 = float('inf')
EPOCHS_P1 = 90
for epoch in range(EPOCHS_P1):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        targets = batch['targets'].to(DEVICE)

        optimizer_p1.zero_grad()
        _, p_report = model(input_ids, attention_mask)
        loss = criterion(p_report, targets)

        loss.backward()
        optimizer_p1.step()
        running_loss += loss.item()

    scheduler_p1.step()
    val_loss = evaluate(model, val_loader, phase=1)

    # Save best Phase 1 checkpoint
    if val_loss < best_val_loss_p1:
        best_val_loss_p1 = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

    if (epoch + 1) % 5 == 0 or epoch == EPOCHS_P1 - 1:
        train_epoch_loss = running_loss / len(train_loader)
        print(f"Phase 1 - Epoch {epoch+1:02d}/{EPOCHS_P1} | Train Loss: {train_epoch_loss:.4f} | Val Loss: {val_loss:.4f}")

print(f"Phase 1 Complete. Best Val Loss: {best_val_loss_p1:.4f}")
for param in model.parameters():
    param.requires_grad = False

for param in model.verification_gate.parameters():
    param.requires_grad = True
# Reload best Phase 1 model state before entering Phase 2
model.load_state_dict(torch.load(MODEL_SAVE_PATH))

# ------------------------------------------------------------------------------
# PHASE 2: TRAIN CROSS-VERIFICATION GATE (Path 2)
# ------------------------------------------------------------------------------
print("\n=== PHASE 2: Training Cross-Verification Gate ===")
optimizer_p2 = AdamW(model.verification_gate.parameters(), lr=1e-4)

best_val_loss_p2 = float('inf')
EPOCHS_P2 = 45

for epoch in range(EPOCHS_P2):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        targets = batch['targets'].to(DEVICE)
        candidate_binary = batch['candidate_binary'].to(DEVICE)
        has_binary_mask = batch['has_binary_mask'].to(DEVICE)

        optimizer_p2.zero_grad()
        _, final_probs = model(input_ids, attention_mask, candidate_binary, has_binary_mask)
        loss = criterion(final_probs, targets)

        loss.backward()
        optimizer_p2.step()
        running_loss += loss.item()

    val_loss = evaluate(model, val_loader, phase=2)

    # Save best overall model checkpoint
    if val_loss < best_val_loss_p2:
        best_val_loss_p2 = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

    if (epoch + 1) % 5 == 0 or epoch == EPOCHS_P2 - 1:
        train_epoch_loss = running_loss / len(train_loader)
        print(f"Phase 2 - Epoch {epoch+1:02d}/{EPOCHS_P2} | Train Loss: {train_epoch_loss:.4f} | Val Loss: {val_loss:.4f}")

print(f"\nTraining Complete! Best model successfully saved to '{MODEL_SAVE_PATH}'.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== PHASE 1: Training Report Extraction Head ===
Phase 1 - Epoch 05/90 | Train Loss: 0.6685 | Val Loss: 0.6561
Phase 1 - Epoch 10/90 | Train Loss: 0.6395 | Val Loss: 0.6313
Phase 1 - Epoch 15/90 | Train Loss: 0.6329 | Val Loss: 0.6182
Phase 1 - Epoch 20/90 | Train Loss: 0.6263 | Val Loss: 0.6137
Phase 1 - Epoch 25/90 | Train Loss: 0.6227 | Val Loss: 0.6117
Phase 1 - Epoch 30/90 | Train Loss: 0.6179 | Val Loss: 0.6114
Phase 1 - Epoch 35/90 | Train Loss: 0.6195 | Val Loss: 0.6112
Phase 1 - Epoch 40/90 | Train Loss: 0.6265 | Val Loss: 0.6098
Phase 1 - Epoch 45/90 | Train Loss: 0.6223 | Val Loss: 0.6072
Phase 1 - Epoch 50/90 | Train Loss: 0.6103 | Val Loss: 0.6025
Phase 1 - Epoch 55/90 | Train Loss: 0.6132 | Val Loss: 0.5984
Phase 1 - Epoch 60/90 | Train Loss: 0.6021 | Val Loss: 0.5948
Phase 1 - Epoch 65/90 | Train Loss: 0.5997 | Val Loss: 0.5914
Phase 1 - Epoch 70/90 | Train Loss: 0.5922 | Val Loss: 0.5914
Phase 1 - Epoch 75/90 | Train Loss: 0.5870 | Val Loss: 0.5911
Phase 1 - Epoch 80/9

In [6]:
# ==============================================================================
# CELL 5B: PSEUDO-LABELING & RETRAINING PIPELINE
# ==============================================================================
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

# 1. Element-wise Masked BCE Loss Function (Handles per-label NaNs)
class ElementwiseMaskedBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss(reduction='none')

    def forward(self, probs, targets):
        # Identify non-NaN values
        valid_mask = ~torch.isnan(targets)
        clean_targets = torch.nan_to_num(targets, nan=0.0)

        # Clamp probabilities for numerical stability
        probs_clamped = torch.clamp(probs, 1e-7, 1.0 - 1e-7)
        loss = self.bce(probs_clamped, clean_targets)

        # Compute loss strictly over valid, pseudo-labeled entries
        masked_loss = (loss * valid_mask.float()).sum() / (valid_mask.float().sum() + 1e-8)
        return masked_loss

# 2. Dataset Handler Supporting Element-wise NaNs
class PseudoLabelDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_train=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['Report'])

        encoded = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        raw_targets = row[LABEL_COLS].values.astype(np.float32)
        targets_tensor = torch.tensor(raw_targets, dtype=torch.float32)
        has_binary_mask = torch.tensor([not np.isnan(raw_targets).all()], dtype=torch.bool)

        # Prepare candidate binaries for Path 2 gate verification
        candidate_binary = torch.nan_to_num(targets_tensor, nan=0.0)

        if self.is_train:
            # Synthetic noise injection for Phase 2 training
            if np.random.rand() < 0.30:
                num_flips = np.random.randint(1, 3)
                flip_indices = np.random.choice(len(LABEL_COLS), size=num_flips, replace=False)
                candidate_binary[flip_indices] = 1.0 - candidate_binary[flip_indices]

        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'targets': targets_tensor,
            'has_binary_mask': has_binary_mask,
            'candidate_binary': candidate_binary
        }

# ------------------------------------------------------------------------------
# STEP 1: GENERATE PSEUDO-LABELS USING PER-CLASS PERCENTILE THRESHOLDS
# ------------------------------------------------------------------------------
print("=== STEP 1: Predicting on 4,349 Unlabeled Reports ===")

# Reload best Phase 1 model weights
model = DualPathMedicalVerifier(model_name="./LaBSE").to(DEVICE)
model.load_state_dict(torch.load("best_dual_path_model.pt", map_location=DEVICE))
model.eval()

unlabeled_df = df[df['ACL'].isnull()].reset_index(drop=True)
unlabeled_ds = PseudoLabelDataset(unlabeled_df, tokenizer, is_train=False)
unlabeled_loader = DataLoader(unlabeled_ds, batch_size=16, shuffle=False)

raw_predictions = []
with torch.no_grad():
    for batch in tqdm(unlabeled_loader, desc="Generating Pseudo-Labels"):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        _, probs = model(input_ids, attention_mask)
        raw_predictions.append(probs.cpu().numpy())

preds_matrix = np.vstack(raw_predictions) # Shape: (4349, 12)

# QUANTILE SELECTION: Select Top 15% as 1.0 and Bottom 15% as 0.0 for EACH class
TOP_PERCENTILE = 85    # Top 15% highest probabilities -> 1.0
BOTTOM_PERCENTILE = 15 # Bottom 15% lowest probabilities -> 0.0

pseudo_matrix = np.full_like(preds_matrix, np.nan)

for col_idx in range(len(LABEL_COLS)):
    col_preds = preds_matrix[:, col_idx]

    high_thresh = np.percentile(col_preds, TOP_PERCENTILE)
    low_thresh = np.percentile(col_preds, BOTTOM_PERCENTILE)

    # Assign pseudo-labels based on per-class distribution
    pseudo_matrix[col_preds >= high_thresh, col_idx] = 1.0
    pseudo_matrix[col_preds <= low_thresh, col_idx] = 0.0

# Calculate coverage
total_entries = pseudo_matrix.size
kept_entries = (~np.isnan(pseudo_matrix)).sum()
coverage = (kept_entries / total_entries) * 100

print(f"\nGenerated Pseudo-Labels across {len(unlabeled_df)} reports.")
print(f"High-Confidence Label Coverage: {kept_entries}/{total_entries} targets ({coverage:.2f}%)")

# Construct Pseudo-Labeled Dataframe
pseudo_df = unlabeled_df[['StudyInstanceUID', 'Report']].copy()
pseudo_df[LABEL_COLS] = pseudo_matrix

# Combine Labeled Ground-Truth (58 rows) with High-Confidence Pseudo-Labels
combined_train_df = pd.concat([train_labeled, pseudo_df], axis=0).reset_index(drop=True)
print(f"Expanded Training Set Size: {len(combined_train_df)} reports.")

# ------------------------------------------------------------------------------
# STEP 2: RETRAIN PATH 1 ON EXPANDED PSEUDO-LABELED DATASET
# ------------------------------------------------------------------------------
print("\n=== STEP 2: Retraining Report Extraction Head (Path 1) on Expanded Dataset ===")

combined_train_ds = PseudoLabelDataset(combined_train_df, tokenizer, is_train=True)
val_ds = PseudoLabelDataset(val_labeled, tokenizer, is_train=False)

combined_train_loader = DataLoader(combined_train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

criterion_masked = ElementwiseMaskedBCELoss()

# Use lower learning rate to preserve pre-trained features while fine-tuning
optimizer_p1_retrain = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
scheduler_p1_retrain = CosineAnnealingLR(optimizer_p1_retrain, T_max=10)

best_val_loss = float('inf')
EPOCHS_RETRAIN_P1 = 10

for epoch in range(EPOCHS_RETRAIN_P1):
    model.train()
    running_loss = 0.0

    for batch in combined_train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        targets = batch['targets'].to(DEVICE)

        optimizer_p1_retrain.zero_grad()
        _, p_report = model(input_ids, attention_mask)
        loss = criterion_masked(p_report, targets)

        loss.backward()
        optimizer_p1_retrain.step()
        running_loss += loss.item()

    scheduler_p1_retrain.step()

    # Validation evaluation on clean ground-truth validation set
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            targets = batch['targets'].to(DEVICE)

            _, probs = model(input_ids, attention_mask)
            val_loss += criterion_masked(probs, targets).item()

    val_loss /= len(val_loader)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_dual_path_model_pseudolabeled.pt")

    train_epoch_loss = running_loss / len(combined_train_loader)
    print(f"Retrain Epoch {epoch+1:02d}/{EPOCHS_RETRAIN_P1} | Train Loss: {train_epoch_loss:.4f} | Val Loss: {val_loss:.4f}")

print(f"\nRetraining Phase 1 Complete. Best Val Loss: {best_val_loss:.4f}")

# ------------------------------------------------------------------------------
# STEP 3: RETRAIN CROSS-VERIFICATION GATE (Path 2)
# ------------------------------------------------------------------------------
print("\n=== STEP 3: Retraining Cross-Verification Gate (Path 2) ===")

model.load_state_dict(torch.load("best_dual_path_model_pseudolabeled.pt", map_location=DEVICE))
optimizer_p2_retrain = AdamW(model.verification_gate.parameters(), lr=5e-5)

EPOCHS_RETRAIN_P2 = 5

for epoch in range(EPOCHS_RETRAIN_P2):
    model.train()
    running_loss = 0.0

    for batch in combined_train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        targets = batch['targets'].to(DEVICE)
        candidate_binary = batch['candidate_binary'].to(DEVICE)

        optimizer_p2_retrain.zero_grad()
        _, final_probs = model(input_ids, attention_mask, candidate_binary)
        loss = criterion_masked(final_probs, targets)

        loss.backward()
        optimizer_p2_retrain.step()
        running_loss += loss.item()

    print(f"Phase 2 Retrain Epoch {epoch+1:02d}/{EPOCHS_RETRAIN_P2} | Train Loss: {running_loss / len(combined_train_loader):.4f}")

# Final Checkpoint Save
torch.save(model.state_dict(), "best_dual_path_model_final.pt")
print("\nPipeline Complete! Saved final model to 'best_dual_path_model_final.pt'.")

=== STEP 1: Predicting on 4,349 Unlabeled Reports ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating Pseudo-Labels: 100%|██████████| 272/272 [00:15<00:00, 17.63it/s]



Generated Pseudo-Labels across 4349 reports.
High-Confidence Label Coverage: 15673/52188 targets (30.03%)
Expanded Training Set Size: 4395 reports.

=== STEP 2: Retraining Report Extraction Head (Path 1) on Expanded Dataset ===
Retrain Epoch 01/10 | Train Loss: 0.2235 | Val Loss: 0.9383
Retrain Epoch 02/10 | Train Loss: 0.0972 | Val Loss: 1.0018
Retrain Epoch 03/10 | Train Loss: 0.0734 | Val Loss: 1.2117
Retrain Epoch 04/10 | Train Loss: 0.0580 | Val Loss: 1.2030
Retrain Epoch 05/10 | Train Loss: 0.0497 | Val Loss: 1.2489
Retrain Epoch 06/10 | Train Loss: 0.0428 | Val Loss: 1.2205
Retrain Epoch 07/10 | Train Loss: 0.0387 | Val Loss: 1.2720
Retrain Epoch 08/10 | Train Loss: 0.0348 | Val Loss: 1.3141
Retrain Epoch 09/10 | Train Loss: 0.0330 | Val Loss: 1.3381
Retrain Epoch 10/10 | Train Loss: 0.0315 | Val Loss: 1.3190

Retraining Phase 1 Complete. Best Val Loss: 0.9383

=== STEP 3: Retraining Cross-Verification Gate (Path 2) ===
Phase 2 Retrain Epoch 01/5 | Train Loss: 0.1202
Phase 2 Re

In [7]:
# ==============================================================================
# CELL 6: INFERENCE PIPELINE (PROBABILITY GENERATOR)
# ==============================================================================
def predict_probabilities(model, tokenizer, report_text, candidate_binaries=None):
    """
    Inference utility for Kaggle evaluation.

    Parameters:
    - report_text (str): Medical report text in any language.
    - candidate_binaries (list/array, optional): 12 binary values (0 or 1).

    Returns:
    - Dictionary mapping each of the 12 classes to its output probability (0.0 to 1.0).
    """
    model.eval()
    encoded = tokenizer(
        report_text,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors='pt'
    )

    input_ids = encoded['input_ids'].to(DEVICE)
    attention_mask = encoded['attention_mask'].to(DEVICE)

    with torch.no_grad():
        if candidate_binaries is not None:
            bin_tensor = torch.tensor([candidate_binaries], dtype=torch.float32).to(DEVICE)
            mask_tensor = torch.tensor([[True]], dtype=torch.bool).to(DEVICE)
            _, probs = model(input_ids, attention_mask, bin_tensor, mask_tensor)
        else:
            _, probs = model(input_ids, attention_mask)

    probs_np = probs.cpu().numpy()[0]
    return dict(zip(LABEL_COLS, probs_np.round(4)))

# Example 1: Pure Text Inference (Path 1)
test_report = "Rotura de menisco interno. Signo de necrosis avascular subcondral."
p_text_only = predict_probabilities(model, tokenizer, test_report)
print("\nPath 1 Output (No binaries provided):\n", p_text_only)

# Example 2: Verification Inference with Input Binaries (Path 2)
candidate_flags = [0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0] # 12 binary inputs
p_verified = predict_probabilities(model, tokenizer, test_report, candidate_binaries=candidate_flags)
print("\nPath 2 Output (With binary verification):\n", p_verified)


Path 1 Output (No binaries provided):
 {'ACL': np.float32(0.5181), 'MCL': np.float32(0.6258), 'Medial Meniscus': np.float32(0.289), 'Lateral Meniscus': np.float32(0.3367), 'Medial OA': np.float32(0.5266), 'Lateral OA': np.float32(0.7111), 'PF OA': np.float32(0.6108), 'Effusion': np.float32(0.138), 'Synovitis': np.float32(0.7209), "Baker's": np.float32(0.8083), 'Contusion': np.float32(0.8147), 'Fracture': np.float32(0.634)}

Path 2 Output (With binary verification):
 {'ACL': np.float32(0.1699), 'MCL': np.float32(0.1929), 'Medial Meniscus': np.float32(0.7196), 'Lateral Meniscus': np.float32(0.1083), 'Medial OA': np.float32(0.2145), 'Lateral OA': np.float32(0.3707), 'PF OA': np.float32(0.2186), 'Effusion': np.float32(0.6706), 'Synovitis': np.float32(0.2746), "Baker's": np.float32(0.3234), 'Contusion': np.float32(0.2754), 'Fracture': np.float32(0.2225)}
